In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier  # Example classifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from xgboost import XGBClassifier
import pickle

import warnings
warnings.filterwarnings("ignore")

In [16]:
cleaned_df=pd.read_csv(r"C:\Users\HP\OneDrive\Documents\Data Science-Self\ML\Mental Health\dataset\cleaned_dataset.csv")
cleaned_df.head(5)

,status,num_of_characters,num_of_sentences,cleaned_statement
0,anxiety,10,1,oh my gosh
1,anxiety,64,2,troubl sleep confus mind restless heart all ou...
2,anxiety,78,2,all wrong back off dear forward doubt stay in ...
3,anxiety,61,1,i have shift my focu to someth els but i am st...
4,anxiety,72,2,i am restless and restless it is been a month ...


In [18]:
cleaned_df.isnull().sum()

status               0
num_of_characters    0
num_of_sentences     0
cleaned_statement    6
dtype: int64

In [20]:
cleaned_df.drop(cleaned_df[cleaned_df['cleaned_statement'].isnull()==True].index,axis=0,inplace=True)

In [22]:
anxiety_df=cleaned_df[cleaned_df['status']=='anxiety']
normal_df=cleaned_df[cleaned_df['status']=='normal']
depression_df=cleaned_df[cleaned_df['status']=='depression']
suicidal_df=cleaned_df[cleaned_df['status']=='suicidal']
stress_df=cleaned_df[cleaned_df['status']=='stress']
bipolar_df=cleaned_df[cleaned_df['status']=='bipolar']
personality_disorder_df=cleaned_df[cleaned_df['status']=='personality disorder']

In [24]:
personality_disorder_df.head()

,status,num_of_characters,num_of_sentences,cleaned_statement
50460,personality disorder,1249,8,is there anyon interest in join a group for av...
50461,personality disorder,479,6,anyon els have noth in common with other peopl...
50462,personality disorder,1857,13,be a ghost would be my ideal form of exist i r...
50463,personality disorder,686,11,i am hurt late i just feel like garbag i have ...
50464,personality disorder,2536,21,i can not cope with my job i work from home as...


In [26]:
print(anxiety_df['status'].value_counts())
print("---------------------------------")
print(normal_df['status'].value_counts())
print("---------------------------------")
print(depression_df['status'].value_counts())
print("---------------------------------")
print(suicidal_df['status'].value_counts())
print("---------------------------------")
print(stress_df['status'].value_counts())
print("---------------------------------")
print(bipolar_df['status'].value_counts())
print("---------------------------------")
print(personality_disorder_df['status'].value_counts())


status
anxiety    3841
Name: count, dtype: int64
---------------------------------
status
normal    16339
Name: count, dtype: int64
---------------------------------
status
depression    15404
Name: count, dtype: int64
---------------------------------
status
suicidal    10650
Name: count, dtype: int64
---------------------------------
status
stress    2587
Name: count, dtype: int64
---------------------------------
status
bipolar    2777
Name: count, dtype: int64
---------------------------------
status
personality disorder    1077
Name: count, dtype: int64


In [28]:
print(cleaned_df['status'].value_counts())

status
normal                  16339
depression              15404
suicidal                10650
anxiety                  3841
bipolar                  2777
stress                   2587
personality disorder     1077
Name: count, dtype: int64


In [30]:
cleaned_df.isnull().sum()

status               0
num_of_characters    0
num_of_sentences     0
cleaned_statement    0
dtype: int64

In [ ]:


# Initialize LabelEncoder
lbl_enc = LabelEncoder()

# Fit and transform labels
cleaned_df['status_encoded'] = lbl_enc.fit_transform(cleaned_df['status'])

# View encoded labels
print("Encoded Labels:", cleaned_df['status_encoded'])

# View label mapping
label_mapping = dict(zip(lbl_enc.classes_, range(len(lbl_enc.classes_))))
print("Label Mapping:", label_mapping)

Encoded Labels: 0        0
1        0
2        0
3        0
4        0
        ..
52676    0
52677    0
52678    0
52679    0
52680    0
Name: status_encoded, Length: 52675, dtype: int32
Label Mapping: {'anxiety': 0, 'bipolar': 1, 'depression': 2, 'normal': 3, 'personality disorder': 4, 'stress': 5, 'suicidal': 6}


In [36]:
# Sample 100 random rows from each DataFrame
seed=101
sample_anxiety = anxiety_df.sample(n=3541, random_state=seed) #3841
sample_normal = normal_df.sample(n=16000, random_state=seed) #16178
sample_depression = depression_df.sample(n=15000, random_state=seed) #15403
sample_suicidal = suicidal_df.sample(n=10000, random_state=seed) #10649
sample_stress = stress_df.sample(n=2587, random_state=seed) #2587
sample_bipolar = bipolar_df.sample(n=2777, random_state=seed) #2777
sample_personality_disorder = personality_disorder_df.sample(n=1077, random_state=seed) #1077

# Concatenate samples vertically (row-wise)
combined_df = pd.concat(
    [sample_anxiety, sample_depression, sample_suicidal, sample_stress, sample_bipolar, sample_personality_disorder]
)




In [38]:
combined_df.drop(combined_df[combined_df['cleaned_statement'].isnull()==True].index,axis=0,inplace=True)

In [40]:
# Assume 'labels' is a column indicating the target for each row in combined_df
x = combined_df[['cleaned_statement','num_of_characters','num_of_sentences']]  # Text features
y = combined_df['status'] # Target variable

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=101)

In [120]:
x_train

,cleaned_statement,num_of_characters,num_of_sentences
51190,do you thinkfeel your therapist get you and yo...,141,3
15635,i am just fill with so much anxieti think abou...,114,1
14971,is it depress when you feel like a zombi like ...,293,6
37516,i get tire of live the same day everi singl day,55,1
8057,i could never leav and not be certain my kid w...,854,11
...,...,...,...
47662,i hit myself when i am upset hi i am f i have ...,1503,20
17909,i feel like i love tattoo becaus they give me ...,1095,10
12743,i have been have depress sinc or so as it seem...,551,7
9454,and i never know when a bad day is come i coul...,568,9


In [42]:
y_train

51190    personality disorder
15635                suicidal
14971              depression
37516                suicidal
8057                 suicidal
                 ...         
47662              depression
17909              depression
12743              depression
9454               depression
24341              depression
Name: status, Length: 27985, dtype: object

In [44]:
combined_df.isnull().sum()

status               0
num_of_characters    0
num_of_sentences     0
cleaned_statement    0
dtype: int64

In [ ]:


# 1. Initialize TF-IDF Vectorizer and fit/transform on the 'tokens' column
vectorizer = TfidfVectorizer(max_features=50000)
x_train_tfidf = vectorizer.fit_transform(x_train['cleaned_statement'])
x_test_tfidf = vectorizer.transform(x_test['cleaned_statement'])

# 2. Extract numerical features
x_train_num = x_train[['num_of_characters', 'num_of_sentences']].values
x_test_num = x_test[['num_of_characters', 'num_of_sentences']].values

# 3. Combine TF-IDF features with numerical features
x_train_combined = hstack([x_train_tfidf, x_train_num])
x_test_combined = hstack([x_test_tfidf, x_test_num])

print('Number of feature words: ', len(vectorizer.get_feature_names_out()))


Number of feature words:  42009


In [52]:
x_train_tfidf

<27985x42009 sparse matrix of type '<class 'numpy.float64'>'
	with 2261161 stored elements in Compressed Sparse Row format>

In [54]:
combined_df

,status,num_of_characters,num_of_sentences,cleaned_statement
35633,anxiety,1069,8,seek postcanc anxieti advic i had cancer at an...
52582,anxiety,890,8,is it normal for an ssri to make you feel like...
51913,anxiety,412,4,today i wa calm and collect hi fella so i wa u...
52048,anxiety,662,6,what actual help your sudden panic attack phys...
34995,anxiety,328,4,use tap water to rins sinus extrem anxiou abou...
...,...,...,...,...
51035,personality disorder,166,2,hikikomori condit have ani of you becom hikiko...
51433,personality disorder,154,1,anyon here in nyc i am a black male an have oc...
50535,personality disorder,576,11,i do not know if i truli deserv to get better ...
51059,personality disorder,319,3,podcast for avoid peopl i am tri to find podca...


In [ ]:
# Apply Random Over-Sampling on the vectorized data
ros = RandomOverSampler(random_state=101)
x_train_resampled, y_train_resampled = ros.fit_resample(x_train_tfidf, y_train)


In [58]:
x_train_resampled

<71988x42009 sparse matrix of type '<class 'numpy.float64'>'
	with 5899516 stored elements in Compressed Sparse Row format>

In [60]:
y_train_resampled

0        personality disorder
1                    suicidal
2                  depression
3                    suicidal
4                    suicidal
                 ...         
71983                suicidal
71984                suicidal
71985                suicidal
71986                suicidal
71987                suicidal
Name: status, Length: 71988, dtype: object

In [ ]:
classifiers = {
    'Bernoulli Naive Bayes': BernoulliNB(alpha=0.1, binarize=0.0),
    'Decision Tree': DecisionTreeClassifier(max_depth=9, min_samples_split=5, random_state=101), 
    'Logistic Regression': LogisticRegression(solver='liblinear', penalty='l1', C=10, random_state=101)
    'XGB': XGBClassifier(learning_rate=0.2, max_depth=7, n_estimators=500, random_state=101) #tree_method='gpu_hist'
}

In [66]:
# Initialize a list to store accuracy scores for each classifier
accuracy_scores = []

# Iterate over each classifier and its name in the classifiers dictionary
for name, clf in classifiers.items():
    clf.fit(x_train_resampled, y_train_resampled)
    y_pred = clf.predict(x_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    
    print("\n")
    print("For", name)
    print("Accuracy:", accuracy)
    
    # Compute the confusion matrix for the predictions
    # 'lbl_enc.classes_' provides the class labels for the confusion matrix and classification report
    #labels = lbl_enc.classes_
    conf_matrix = confusion_matrix(y_test, y_pred)
    #print(classification_report(y_test, y_pred, target_names=labels))
    
    # Plot the confusion matrix using a heatmap
    # Annotate each cell with the numeric value of the confusion matrix
    #sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)

    #plt.xlabel('Predicted')  # Label for x-axis
    #plt.ylabel('Actual')     # Label for y-axis
    #plt.title(f'Confusion Matrix for {name}')  # Title for the heatmap
    #plt.show()  # Display the heatmap
    
    # Append the accuracy score to the list
    accuracy_scores.append(accuracy)





For Bernoulli Naive Bayes
Accuracy: 0.5113620122909819


For Decision Tree
Accuracy: 0.41460625982563953


For Logistic Regression
Accuracy: 0.6750035729598399


In [141]:

# Train a model (e.g., RandomForestClassifier)
model = RandomForestClassifier(max_depth=40, min_samples_split=5, random_state=101)
model.fit(x_train_resampled, y_train_resampled)

# Predict using the trained model
y_pred = model.predict(x_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")


KeyboardInterrupt



In [ ]:

# Wrap the base model with BaggingClassifier
bagging_model = BaggingClassifier(estimator=model, n_estimators=25, random_state=101)

# Train the bagging model
bagging_model.fit(x_train_resampled, y_train_resampled)

# Predict using the bagging model
y_pred_bagging = bagging_model.predict(x_test_tfidf)

# Calculate accuracy
accuracy_bagging = accuracy_score(y_test, y_pred_bagging)
print(f"Bagging Model Accuracy: {accuracy_bagging:.4f}")

Bagging Model Accuracy: 0.7003


In [62]:
print(confusion_matrix(y_test, y_pred_bagging))

[[ 600    7   50    1   20   15]
 [  19  419   54    0   19   23]
 [ 110   66 1933    3   50  840]
 [  10    0   68  111    5   28]
 [  62    6  100    1  303   45]
 [  44   25  400    0   26 1534]]


In [70]:
log_reg_model=LogisticRegression(solver='liblinear', penalty='l1', C=10, random_state=101)
log_reg_model.fit(x_train_resampled, y_train_resampled)

# Predict using the trained model
y_pred_log_reg = log_reg_model.predict(x_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_log_reg)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.6750


In [76]:
print(confusion_matrix(y_test, y_pred_log_reg))

[[ 542   10   66   11   42   22]
 [  26  404   55    3   14   32]
 [  71   64 2024   46  115  682]
 [   7    5   41  146    8   15]
 [  53   23   82   16  300   43]
 [  24   40  577   14   67 1307]]


In [74]:
import pickle
#pickle.dump(vectorizer,open('models/vectorizer.pkl','wb'))
#pickle.dump(bagging_model,open('models/classifier.pkl','wb'))
pickle.dump(log_reg_model,open(r'C:\Users\HP\OneDrive\Documents\Data Science-Self\ML\Mental Health\models\log_reg_classifier_with_resampling.pkl','wb'))

TfidfVectorizer(max_features=50000)

0.461978273299028


Best Parameters:
{'C': 1}
Accuracy Score:
0.6858504049547404


Decision Tree Accuracy: 0.5773
